In [1]:
import pandas as pd
import numpy as np

In [2]:
questions = pd.read_csv("C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/VLAT Questions.csv")
questions.head()

,id,dropped,vis,item,question:,option:,correct
0,1,no,VLAT_a,a_1,What was the price of a barrel of oil in Febru...,$39.72; $57.36; $48.90; $62.85,$48.90
1,2,no,VLAT_a,a_2,In which month was the price of a barrel of oi...,April; June; September; December,April
2,3,no,VLAT_a,a_3,What was the price range of a barrel of oil in...,$48.36 - $60.95; $37.75 - $63.40; $31.04 - $60...,$37.75 - $63.40
3,4,no,VLAT_a,a_4,"Over the course of the second half of 2015, th...",rising; falling; staying,rising
4,5,no,VLAT_a,a_5,About how much did the price of a barrel of oi...,$4.56; $14.63; $23.55; $45.56,$23.55


In [3]:
dropped_questions = set(questions[questions['dropped'] == "yes"]['id'].to_numpy())
dropped_questions

{13, 24, 26, 30, 39, 43, 50, 58}

In [4]:
questions['num_options'] = questions['option:'].apply(lambda x: len(x.split(";")))
questions.tail(3)

,id,dropped,vis,item,question:,option:,correct,num_options
58,59,no,VLAT_l,l_2,For which website was the number of unique vis...,Facebook; Amazon; Bing; Google,Google,4
59,60,no,VLAT_l,l_3,The number of unique visitors for Amazon was m...,True; False,TRUE,2
60,61,no,VLAT_l,l_4,Samsung is nested in the Computer category.,True; False,TRUE,2


In [5]:
questions = questions.set_index("id")

In [6]:
questions_meta = pd.read_csv("C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/VLAT Questions Metadata.csv")
questions_meta = questions_meta.set_index("id").fillna("")
questions_meta.tail(3)

,vis,task,subtask,difficulty,discrimination
id,,,,,
59,treemap,find extremum,relative value,moderate,high
60,treemap,make comparisons,relative value,hard,high
61,treemap,identify hierarchical structure,,easy,medium


In [7]:
results_1 = pd.read_csv("C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Results/1_Claude 3.7_Final Results with Explanation/VLAT_1741461195.csv")
results_2 = pd.read_csv("C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Results/1_Claude 3.7_Final Results with Explanation/VLAT_1741463278.csv")
results_3 = pd.read_csv("C:/Users/amitc/OneDrive/Desktop/New folder (7)/New_VLAT_Graphs/Results/1_Claude 3.7_Final Results with Explanation/VLAT_1741465715.csv")

In [8]:
results = results_1.merge(results_2, on='id', suffixes=("", "_2"))
results = results.merge(results_3, on='id', suffixes=("", "_3"))
results = results[~results['id'].isin(dropped_questions)]
results = results.set_index("id")

In [9]:
len(results)

53

In [10]:
results.head()

,response,time,correct_bool,response_2,time_2,correct_bool_2,response_3,time_3,correct_bool_3
id,,,,,,,,,
1,$48.90,9.499368,True,$48.90,9.112734,True,$48.90,9.420676,True
2,April,8.969380,True,April,8.199355,True,April,9.585690,True
3,$37.75 - $63.40,12.422285,True,$37.75 - $63.40,11.157850,True,$37.75 - $63.40,12.266117,True
4,rising,9.261708,True,rising,8.948347,True,rising,9.358780,True
5,$23.55,9.524704,True,$23.55,9.675365,True,$23.55,9.655496,True


In [11]:
responses = results[['response', 'response_2', 'response_3']]
responses.head()

,response,response_2,response_3
id,,,
1,$48.90,$48.90,$48.90
2,April,April,April
3,$37.75 - $63.40,$37.75 - $63.40,$37.75 - $63.40
4,rising,rising,rising
5,$23.55,$23.55,$23.55


In [12]:
agreements = np.logical_and(responses['response'] == responses['response_2'], responses['response_2'] == responses['response_3'])
responses[~agreements]

,response,response_2,response_3
id,,,
9,1,2 countries,2 countries
15,FALSE,TRUE,FALSE
22,314,310,314


In [13]:
# Correct way to combine multiple logical conditions
omissions = np.logical_or(
    np.logical_or(responses['response'] == "Omit", responses['response_2'] == "Omit"),
    responses['response_3'] == "Omit"
)

# Then filter
responses[omissions]

,response,response_2,response_3
id,,,


In [14]:
scoring = results.copy()
scoring["correct_bool"] = np.where(scoring["response"] == "Omit", "Omit", scoring["correct_bool"])
scoring["correct_bool_2"] = np.where(scoring["response_2"] == "Omit", "Omit", scoring["correct_bool_2"])
scoring["correct_bool_3"] = np.where(scoring["response_3"] == "Omit", "Omit", scoring["correct_bool_3"])
scoring.head(6)

,response,time,correct_bool,response_2,time_2,correct_bool_2,response_3,time_3,correct_bool_3
id,,,,,,,,,
1,$48.90,9.499368,True,$48.90,9.112734,True,$48.90,9.420676,True
2,April,8.969380,True,April,8.199355,True,April,9.585690,True
3,$37.75 - $63.40,12.422285,True,$37.75 - $63.40,11.157850,True,$37.75 - $63.40,12.266117,True
4,rising,9.261708,True,rising,8.948347,True,rising,9.358780,True
5,$23.55,9.524704,True,$23.55,9.675365,True,$23.55,9.655496,True
6,17 Mbps,9.579563,True,17 Mbps,9.515645,True,17 Mbps,9.739185,True


In [15]:
def getScore(correct_arr):
    score = 0
    for i, item in enumerate(correct_arr):
        if item == "True":
            score += 1
        elif item == "False":
            score -= 1/(questions.loc[i,"num_options"] - 1)
    return score

In [16]:
for col in ["correct_bool", "correct_bool_2", "correct_bool_3"]:
    score = sum(scoring[col] == "True")
    print(col+":", round(score, 2), "(raw)")

correct_bool: 50 (raw)
correct_bool_2: 51 (raw)
correct_bool_3: 50 (raw)


In [17]:
for col in ["correct_bool", "correct_bool_2", "correct_bool_3"]:
    score = getScore(scoring[col].to_numpy())
    print(col+":", round(score, 2))

correct_bool: 49.0
correct_bool_2: 50.33
correct_bool_3: 49.0


In [18]:
# Chain logical_and operations correctly
scoring["correct_bool_consensus"] = np.where(
    # First logical_and chain for the condition
    np.logical_and(
        np.logical_and(scoring["response"] == "Omit", scoring["response_2"] == "Omit"),
        scoring["response_3"] == "Omit"
    ), 
    "Omit",
    # Second logical_and chain for the value
    np.logical_and(
        np.logical_and(scoring["correct_bool"] == "True", scoring["correct_bool_2"] == "True"),
        scoring["correct_bool_3"] == "True"
    )
)

# View the results
scoring[["correct_bool", "correct_bool_2", "correct_bool_3", "correct_bool_consensus"]].head(10)

,correct_bool,correct_bool_2,correct_bool_3,correct_bool_consensus
id,,,,
1,True,True,True,True
2,True,True,True,True
3,True,True,True,True
4,True,True,True,True
5,True,True,True,True
6,True,True,True,True
7,False,False,False,False
8,True,True,True,True
9,False,False,False,False


In [19]:
# First convert string "True" to actual boolean values if needed
scoring["correct_bool_consensus"] = scoring["correct_bool_consensus"].astype(str) == "True"

# Now calculate the raw score (sum of True values)
consensus_score_raw = scoring["correct_bool_consensus"].sum()
print(round(consensus_score_raw, 2), "(raw)")

# If getScore is a custom function, make sure it works with boolean arrays
# If it's not defined, you might need to implement it
consensus_score = getScore(scoring["correct_bool_consensus"].to_numpy())
print(round(consensus_score, 2))

50 (raw)
0


In [20]:
correct_bool_consensus = scoring[["correct_bool_consensus"]]
questions_meta_concensus_ans = correct_bool_consensus.join(questions_meta)
questions_meta_concensus_ans.head(5)

,correct_bool_consensus,vis,task,subtask,difficulty,discrimination
id,,,,,,
1,True,line chart,retrieve value,,easy,low
2,True,line chart,find extremum,,easy,low
3,True,line chart,determine range,,moderate,high
4,True,line chart,find correlations/trends,,easy,low
5,True,line chart,make comparisons,,moderate,high


In [21]:
for col in ['vis', 'task', 'difficulty']:
    vals = questions_meta_concensus_ans[questions_meta_concensus_ans['correct_bool_consensus'] == "True"][col].value_counts()
    print(vals, "\n")

Series([], Name: vis, dtype: int64) 

Series([], Name: task, dtype: int64) 

Series([], Name: difficulty, dtype: int64) 



In [22]:
for col in ['vis', 'task', 'difficulty']:
    vals = questions_meta_concensus_ans[questions_meta_concensus_ans['correct_bool_consensus'] == "False"][col].value_counts()
    print(vals, "\n")

Series([], Name: vis, dtype: int64) 

Series([], Name: task, dtype: int64) 

Series([], Name: difficulty, dtype: int64) 



In [23]:
for col in ['vis', 'task', 'difficulty']:
    vals = questions_meta_concensus_ans[questions_meta_concensus_ans['correct_bool_consensus'] == "Omit"][col].value_counts()
    print(vals, "\n")

Series([], Name: vis, dtype: int64) 

Series([], Name: task, dtype: int64) 

Series([], Name: difficulty, dtype: int64) 



In [24]:
questions_meta_concensus_ans[["vis", "correct_bool_consensus"]].value_counts(sort=False)

vis                     correct_bool_consensus
100% stacked bar chart  True                      3
area chart              True                      4
bar chart               False                     2
                        True                      2
bubble chart            True                      7
choropleth map          True                      3
histogram               True                      3
line chart              True                      5
pie chart               True                      3
scatterplot             True                      7
stacked area chart      True                      6
stacked bar chart       False                     1
                        True                      4
treemap                 True                      3
dtype: int64

In [25]:
questions_meta_concensus_ans[["task", "correct_bool_consensus"]].value_counts(sort=False)

task                             correct_bool_consensus
determine range                  True                       5
find anomalies                   True                       2
find clusters                    True                       2
find correlations/trends         True                       5
find extremum                    False                      1
                                 True                      11
identify hierarchical structure  True                       1
make comparisons                 False                      2
                                 True                      11
retrieve value                   True                      13
dtype: int64

In [26]:
questions_meta_concensus_ans[["difficulty", "correct_bool_consensus"]].value_counts(sort=False)

difficulty  correct_bool_consensus
easy        False                      1
            True                      16
hard        False                      2
            True                      15
moderate    True                      19
dtype: int64

In [27]:
# Correct, hard
questions_meta_concensus_ans[np.logical_and(questions_meta_concensus_ans['correct_bool_consensus'] == "True",
                                            questions_meta_concensus_ans['difficulty'] == "hard")]

,correct_bool_consensus,vis,task,subtask,difficulty,discrimination
id,,,,,,
